In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch.optim as optim
from tqdm import tqdm
import os


In [2]:
# Пытаемся подключить диск только если мы в Colab
try:
    from google.colab import drive
    if os.path.exists('/content'):
        if not os.path.isdir('/content/drive'):
            drive.mount('/content/drive')
except ImportError:
    # Если мы локально, библиотеки google.colab нет, и это нормально
    print("Запуск локально: модуль google.colab не найден, пропускаем монтирование.")


Mounted at /content/drive


In [3]:
# Определяем корень проекта (BASE_DIR)
if os.path.exists('/content/drive/MyDrive/'):
    # Путь к папке на Диске (проверьте название 'my_project' или 'CLASSIFICATION')
    BASE_DIR = '/content/drive/MyDrive/CLASSIFICATION' 
else:
    # Локально или на GitHub корень — это текущая папка
    BASE_DIR = '.'

# Ко всем путям приклеиваем BASE_DIR
PATHS = {
    'train': os.path.join(BASE_DIR, 'dataset/train'),
    'validation':   os.path.join(BASE_DIR, 'dataset/validation'),
    'best_model': os.path.join(BASE_DIR, 'best_model.pth')
}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1. Настройка устройства

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### 2. Трансформации (Добавлена нормализация и базовая аугментация)

In [5]:
stats = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # Случайный поворот для лучшего обучения
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

### 3. Загрузка данных

In [6]:
train_data_dir = PATHS['train']
val_data_dir = PATHS['validation']

train_dataset = ImageFolder(train_data_dir, transform=train_transforms)
val_dataset = ImageFolder(val_data_dir, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

### 4. Инициализация модели

In [7]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

num_classes = len(train_dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device) # Переносим модель на GPU/CPU

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 205MB/s]


### 5. Оптимизация и функция потерь

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) # 0.01, 0.001, 0.0001

### 6. Цикла обучения

In [9]:
# Cохранение модели в Google Drive
save_path = PATHS['best_model']

# Инициализация лучшей точности
best_val_acc = 0.0

epochs = 30

for epoch in range(epochs):
    # Тренировка
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    # Валидация
    model.eval()
    correct = 0
    total = 0
    val_loss = 0.0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    current_val_acc = 100 * correct / total
    avg_train_loss = running_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    print(f"\n[Epoch {epoch+1}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {current_val_acc:.2f}%")
 

    # Сохранение лучшей модели
    if current_val_acc > best_val_acc:
        best_val_acc = current_val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'accuracy': best_val_acc,
        }, save_path)
        print(f"--> Модель сохранена! Новая лучшая точность: {best_val_acc:.2f}%")
    print()
print(f"Обучение завершено! Лучшая точность: {best_val_acc:.2f}%")

Epoch 1/30: 100%|██████████| 12/12 [08:11<00:00, 41.00s/it, loss=0.4588]



[Epoch 1] Train Loss: 0.8426 | Val Loss: 26.0343 | Val Acc: 37.72%
--> Модель сохранена! Новая лучшая точность: 37.72%



Epoch 2/30: 100%|██████████| 12/12 [00:06<00:00,  1.94it/s, loss=0.0799]



[Epoch 2] Train Loss: 0.4390 | Val Loss: 4.2640 | Val Acc: 52.63%
--> Модель сохранена! Новая лучшая точность: 52.63%



Epoch 3/30: 100%|██████████| 12/12 [00:05<00:00,  2.12it/s, loss=0.7027]


...


[Epoch 28] Train Loss: 0.0657 | Val Loss: 0.5839 | Val Acc: 84.21%



Epoch 29/30: 100%|██████████| 12/12 [00:05<00:00,  2.35it/s, loss=0.0075]



[Epoch 29] Train Loss: 0.1432 | Val Loss: 1.8030 | Val Acc: 68.42%



Epoch 30/30: 100%|██████████| 12/12 [00:05<00:00,  2.15it/s, loss=0.0260]



[Epoch 30] Train Loss: 0.1347 | Val Loss: 0.5683 | Val Acc: 83.33%

Обучение завершено! Лучшая точность: 86.84%
